# Exercise 8

Bootstrap exercise.

## Imports

Just the normal setup stuff.

In [ ]:
import math
import os
import random
import statistics

try:
    import numpy as np
except ImportError:
    np = None

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

random.seed(12345)
PLOT_DIR = "Exercise8/pics"
os.makedirs(PLOT_DIR, exist_ok=True)


def save_fig(fig, filename):
    if plt is None:
        print("no matplotlib:", filename)
        return
    path = os.path.join(PLOT_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("saved", path)


print("numpy:", np is not None)
print("matplotlib:", plt is not None)


## Helpers

Mean, variance, bootstrap resampling, percentile interval, that kind of thing.

In [ ]:
def sample_mean(xs):
    return sum(xs) / len(xs)


def sample_variance(xs):
    n = len(xs)
    xbar = sample_mean(xs)
    return sum((x - xbar) ** 2 for x in xs) / (n - 1)


def sample_variance_mle(xs):
    n = len(xs)
    xbar = sample_mean(xs)
    return sum((x - xbar) ** 2 for x in xs) / n


def median(xs):
    return statistics.median(xs)


def bootstrap_sample(xs):
    return [random.choice(xs) for _ in range(len(xs))]


def bootstrap_replicates(xs, statistic, B=100):
    reps = []
    for _ in range(B):
        samp = bootstrap_sample(xs)
        reps.append(statistic(samp))
    return reps


def bootstrap_variance(replicates):
    return sample_variance(replicates)


def percentile_interval(replicates, alpha=0.05):
    ys = sorted(replicates)
    m = len(ys)
    low_idx = max(0, int((alpha / 2) * m))
    high_idx = min(m - 1, int((1 - alpha / 2) * m))
    return ys[low_idx], ys[high_idx]


def quantile(xs, q):
    ys = sorted(xs)
    if not ys:
        return None
    idx = min(len(ys) - 1, max(0, int(q * (len(ys) - 1))))
    return ys[idx]


def print_table(rows, headers):
    widths = []
    for i, h in enumerate(headers):
        widths.append(max(len(h), max(len(str(r[i])) for r in rows)))
    print(" | ".join(headers[i].ljust(widths[i]) for i in range(len(headers))))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(str(row[i]).ljust(widths[i]) for i in range(len(headers))))


def bootstrap_median_variance(sample, B=100):
    med = median(sample)
    reps = bootstrap_replicates(sample, median, B=B)
    return med, bootstrap_variance(reps), reps


def bootstrap_mean_variance(sample, B=100):
    mean_val = sample_mean(sample)
    reps = bootstrap_replicates(sample, sample_mean, B=B)
    return mean_val, bootstrap_variance(reps), reps


## Quick note

The idea is pretty simple really: resample from the data with replacement, recompute the stat, and use the spread of those bootstrap values as a rough variance/uncertainty estimate.

## Part 1 - Ross 13

Here I estimate the probability using bootstrap sample means instead of the unknown true mean.

In [ ]:
x13 = [56, 101, 78, 67, 93, 87, 64, 72, 80, 69]
n13 = len(x13)
a = -5
b = 5
B13 = 10_000
xbar13 = sample_mean(x13)

boot_diffs = []
count_inside = 0
for _ in range(B13):
    samp = bootstrap_sample(x13)
    diff = sample_mean(samp) - xbar13
    boot_diffs.append(diff)
    if a < diff < b:
        count_inside += 1

p_hat = count_inside / B13
mc_se = math.sqrt(p_hat * (1 - p_hat) / B13)
ci_low = p_hat - 1.96 * mc_se
ci_high = p_hat + 1.96 * mc_se

print("sample mean:", round(xbar13, 4))
print("p_hat:", round(p_hat, 5))
print("Monte Carlo SE:", round(mc_se, 6))
print("approx 95% CI for p_hat:", (round(ci_low, 5), round(ci_high, 5)))


In [ ]:
if plt is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(boot_diffs, bins=40, color="steelblue", edgecolor="black", alpha=0.8)
    ax.axvline(a, color="darkred", linestyle="--", label="a")
    ax.axvline(b, color="darkgreen", linestyle="--", label="b")
    ax.set_title("Exercise 13 bootstrap: xbar* - xbar")
    ax.set_xlabel("xbar* - xbar")
    ax.set_ylabel("count")
    ax.legend()
    ax.grid(alpha=0.2)
    save_fig(fig, "ex13_bootstrap_hist.png")
    plt.close(fig)


So here `p` is just the fraction of bootstrap means that land in the interval around the original sample mean.

## Part 2 - Ross 15

Now its the bootstrap variance of the usual sample variance `S^2`.

In [ ]:
x15 = [5, 4, 9, 6, 21, 17, 11, 20, 7, 10, 21, 15, 13, 16, 8]
n15 = len(x15)
B15 = 10_000

mean15 = sample_mean(x15)
s2_15 = sample_variance(x15)
boot_s2 = bootstrap_replicates(x15, sample_variance, B=B15)
boot_mean_s2 = sample_mean(boot_s2)
boot_var_s2 = bootstrap_variance(boot_s2)
boot_se_s2 = math.sqrt(boot_var_s2)
pi_s2 = percentile_interval(boot_s2, alpha=0.05)

print("sample mean:", round(mean15, 4))
print("sample variance S^2:", round(s2_15, 4))
print("bootstrap mean of S^2:", round(boot_mean_s2, 4))
print("bootstrap Var(S^2):", round(boot_var_s2, 4))
print("bootstrap SE(S^2):", round(boot_se_s2, 4))
print("95% percentile interval:", tuple(round(x, 4) for x in pi_s2))


In [ ]:
if plt is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(boot_s2, bins=40, color="darkorange", edgecolor="black", alpha=0.8)
    ax.axvline(s2_15, color="black", linestyle="--", label="original S^2")
    ax.set_title("Bootstrap values of S^2")
    ax.set_xlabel("S^2")
    ax.set_ylabel("count")
    ax.legend()
    ax.grid(alpha=0.2)
    save_fig(fig, "ex15_bootstrap_var_s2.png")
    plt.close(fig)


This one is basically just looking at how much `S^2` moves around across the resamples.

## Part 3 - Pareto sample

This is the interesting part bc the tail is super heavy, so the mean can get pushed around a lot.

In [ ]:
def pareto_sample(n, beta=1.0, alpha=1.05):
    xs = []
    for _ in range(n):
        u = random.random()
        xs.append(beta * (u ** (-1 / alpha)))
    return xs


n_p = 200
beta = 1.0
alpha = 1.05
B_p = 100

pareto_x = pareto_sample(n_p, beta=beta, alpha=alpha)
pareto_mean = sample_mean(pareto_x)
pareto_median = median(pareto_x)
pareto_min = min(pareto_x)
pareto_max = max(pareto_x)
q90 = quantile(pareto_x, 0.90)
q95 = quantile(pareto_x, 0.95)
q99 = quantile(pareto_x, 0.99)

mean_val, boot_var_mean, boot_means = bootstrap_mean_variance(pareto_x, B=B_p)
med_val, boot_var_median, boot_medians = bootstrap_median_variance(pareto_x, B=B_p)
se_mean = math.sqrt(boot_var_mean)
se_median = math.sqrt(boot_var_median)
var_ratio = boot_var_mean / boot_var_median if boot_var_median > 0 else float("inf")

print("sample mean:", round(pareto_mean, 4))
print("sample median:", round(pareto_median, 4))
print("min/max:", round(pareto_min, 4), round(pareto_max, 4))
print("q90, q95, q99:", round(q90, 4), round(q95, 4), round(q99, 4))
print("bootstrap Var(mean):", round(boot_var_mean, 4))
print("bootstrap Var(median):", round(boot_var_median, 4))
print("bootstrap SE(mean):", round(se_mean, 4))
print("bootstrap SE(median):", round(se_median, 4))
print("variance ratio:", round(var_ratio, 4))


In [ ]:
if plt is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(pareto_x, bins=40, color="slateblue", edgecolor="black", alpha=0.8)
    ax.set_xlim(0, min(100, max(pareto_x)))
    ax.set_title("Pareto sample")
    ax.set_xlabel("x")
    ax.set_ylabel("count")
    ax.grid(alpha=0.2)
    save_fig(fig, "pareto_sample_hist.png")
    plt.close(fig)


In [ ]:
if plt is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(boot_means, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
    axes[0].set_title("Bootstrap means")
    axes[0].grid(alpha=0.2)
    axes[1].hist(boot_medians, bins=20, color="seagreen", edgecolor="black", alpha=0.8)
    axes[1].set_title("Bootstrap medians")
    axes[1].grid(alpha=0.2)
    save_fig(fig, "pareto_bootstrap_mean_vs_median.png")
    plt.close(fig)


In [ ]:
if plt is not None:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.boxplot([boot_means, boot_medians], tick_labels=["mean", "median"])
    ax.set_title("Bootstrap replicates")
    ax.grid(alpha=0.2)
    save_fig(fig, "pareto_bootstrap_boxplot.png")
    plt.close(fig)


Heavy tail shows up pretty clearly here. The mean is way more shakey than the median.

## Optional repeated Pareto check

I repeated it a few times just to see if that pattern keeps showing up.

In [ ]:
repeated_rows = []
for rep in range(1, 21):
    xs = pareto_sample(n_p, beta=beta, alpha=alpha)
    m1, v1, _ = bootstrap_mean_variance(xs, B=B_p)
    m2, v2, _ = bootstrap_median_variance(xs, B=B_p)
    repeated_rows.append([
        rep,
        round(sample_mean(xs), 3),
        round(median(xs), 3),
        round(v1, 3),
        round(v2, 3),
    ])

print_table(repeated_rows[:10], ["rep", "sample mean", "sample median", "boot var mean", "boot var median"])
print("... showing first 10 only")


## Tables

Putting the main numbers together here so its easier to copy into the report.

In [ ]:
table1 = [[n13, B13, round(xbar13, 4), a, b, round(p_hat, 5), round(mc_se, 6)]]
print("Exercise 13")
print_table(table1, ["n", "B", "sample mean", "a", "b", "p_hat", "MC SE"])

table2 = [[
    n15,
    B15,
    round(s2_15, 4),
    round(boot_mean_s2, 4),
    round(boot_var_s2, 4),
    round(boot_se_s2, 4),
    tuple(round(x, 4) for x in pi_s2),
]]
print("\nExercise 15")
print_table(table2, ["n", "B", "orig S^2", "boot mean S^2", "boot Var(S^2)", "boot SE(S^2)", "percentile interval"])

table3 = [[
    n_p,
    beta,
    alpha,
    B_p,
    round(pareto_mean, 4),
    round(pareto_median, 4),
    round(boot_var_mean, 4),
    round(boot_var_median, 4),
    round(se_mean, 4),
    round(se_median, 4),
    round(var_ratio, 4),
]]
print("\nPareto part")
print_table(
    table3,
    ["n", "beta", "alpha", "B", "sample mean", "sample median", "Var(mean)", "Var(median)", "SE(mean)", "SE(median)", "ratio"],
)


## End

Exercise 13 and 15 both use the same bootstrap idea, just on diff statistics. For the Pareto sample the mean was clearly less stable than the median, which fits the heavy-tail story pretty well.